# Axon Training Pipeline — Colab Notebook

**From floor plan to fabrication. Any PDF. Your products. Zero guesswork.**

This notebook runs the full Axon training pipeline on Google Colab with GPU acceleration:

1. **Setup** — Pull from GitHub, install dependencies, mount Google Drive
2. **Data** — Upload or link training datasets
3. **MPM Pre-training** — Masked Primitive Modeling (self-supervised)
4. **SFT Fine-tuning** — Supervised fine-tuning on labeled data
5. **GRPO Annealing** — Quality annealing with group relative policy optimization
6. **DRL Training** — Deep Reinforcement Learning for panelization + placement
7. **Evaluation** — Run benchmarks on trained models

**Requirements:** Colab GPU runtime (RTX 6000 Pro Blackwell / 48GB VRAM). Go to *Runtime > Change runtime type > GPU*.

## 1. Setup

Pull the latest code from GitHub, install dependencies, and mount Google Drive for persistent checkpoint storage.

In [ ]:
# Pull latest from GitHub (clone on first run, pull on subsequent runs)
import os

if os.path.exists("Axon"):
    %cd Axon
    !git pull origin main
else:
    !git clone https://github.com/tylermark/Axon.git
    %cd Axon

# Install only what training needs (Colab already has torch, numpy, PIL, etc.)
!pip install -q pydantic pymupdf scipy networkx wandb gymnasium sb3-contrib stable-baselines3 2>/dev/null

# Verify GPU availability
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)
    gpu_mem_gb = gpu_mem / 1e9
    print(f"GPU device:      {gpu_name}")
    print(f"GPU memory:      {gpu_mem_gb:.1f} GB")
    if gpu_mem_gb >= 40:
        print("RTX 6000 Pro detected — using large batch sizes.")
    else:
        print("WARNING: Expected RTX 6000 Pro (48GB). Batch sizes may need adjustment.")

In [ ]:
# Mount Google Drive for persistent checkpoints and data
from google.colab import drive
drive.mount("/content/drive")

# Persistent directories on Google Drive
DRIVE_ROOT = "/content/drive/MyDrive/axon"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
DATA_DIR = f"{DRIVE_ROOT}/datasets"
LOG_DIR = f"{DRIVE_ROOT}/logs"

import os
for d in [CKPT_DIR, DATA_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"  {d}")

print(f"\nCheckpoints will persist at: {CKPT_DIR}")
print(f"Datasets expected at:        {DATA_DIR}")

## 2. Upload Datasets

Your datasets are stored locally at `C:\Users\tyler\Axon\datasets\`. Upload them to Google Drive at `axon/datasets/` so Colab can access them.

**One-time setup:**
1. Open Google Drive in your browser
2. Create folder: `My Drive/axon/datasets/`
3. Upload these folders from your local `datasets/` directory:
   - `archcad400k/` (primary — vector floor plans)
   - `FLOORPLANCAD/` (CAD drawings)
   - `ResPlan/` (structured room graphs)
   - `MLSTRUCT-FP_v1/` (raster images)

After uploading once, they persist on Drive across all Colab sessions.

In [ ]:
# Link Drive datasets into the working directory
import os
from pathlib import Path

local_data = Path("datasets")
local_data.mkdir(exist_ok=True)

drive_data = Path(DATA_DIR)

# Check which datasets are available
expected = ["archcad400k", "FLOORPLANCAD", "ResPlan", "MLSTRUCT-FP_v1"]
found = []
missing = []

if drive_data.exists():
    for name in expected:
        src = drive_data / name
        target = local_data / name
        if src.exists():
            if not target.exists():
                os.symlink(str(src), str(target))
            found.append(name)
            print(f"  [OK] {name}")
        else:
            missing.append(name)
            print(f"  [MISSING] {name} — upload to {DATA_DIR}/{name}")
else:
    missing = expected
    print(f"Drive datasets folder not found at {DATA_DIR}")
    print("Run cell 3 first (mount Drive), then upload datasets.")

print(f"\nFound: {len(found)}/{len(expected)} datasets")
if missing:
    print(f"Missing: {', '.join(missing)}")
    print(f"\nUpload from your local C:\\Users\\tyler\\Axon\\datasets\\ to Google Drive at My Drive/axon/datasets/")

## 3. MPM Pre-training

**Masked Primitive Modeling** — self-supervised pre-training that learns structural
representations by masking 80% of vector primitives and reconstructing them.

This is Phase A of the training pipeline. Takes ~1-2 hours on the RTX 6000 Pro with large batch sizes.

In [ ]:
from src.training.mpm import MPMPreTrainer, MPMConfig
from src.training.tracking import ExperimentTracker, CheckpointManager

# Configure MPM pre-training (large batch sizes for RTX 6000 Pro 48GB)
mpm_config = MPMConfig(
    mask_ratio_low=0.75,
    mask_ratio_high=0.85,
    num_epochs=100,
    batch_size=64,  # 48GB VRAM allows large batches
    learning_rate=1e-4,
    checkpoint_dir=f"{CKPT_DIR}/mpm",
    device="cuda" if torch.cuda.is_available() else "cpu",
    wandb_project="axon-mpm",
    wandb_run_name="mpm-pretrain-colab",
)

mpm_ckpt = CheckpointManager(
    checkpoint_dir=f"{CKPT_DIR}/mpm",
    max_checkpoints=5,
)

print(f"MPM config: mask=[{mpm_config.mask_ratio_low}, {mpm_config.mask_ratio_high}], "
      f"epochs={mpm_config.num_epochs}, batch={mpm_config.batch_size}, lr={mpm_config.learning_rate}")
print(f"Checkpoints: {CKPT_DIR}/mpm")

In [ ]:
# Run MPM pre-training
from src.training.mpm import MPMModel
from src.training.data_engine import ArchCAD400KDataset, CombinedFloorPlanDataset

# Load dataset
dataset = ArchCAD400KDataset(data_root="datasets/")
print(f"Dataset: {len(dataset)} samples")

# Create model
model = MPMModel(mpm_config)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model: {param_count:,} parameters")

# Create trainer and run
trainer = MPMPreTrainer(model=model, dataset=dataset, config=mpm_config)
trainer.train()

print("MPM pre-training complete. Checkpoint saved to Google Drive.")

## 4. SFT Fine-tuning

**Supervised Fine-Tuning** — fine-tune the pre-trained model on labeled floor plan
datasets (archcad400k, FloorPlanCAD).

This is Phase B of the training pipeline. Requires an MPM checkpoint from Step 3.

In [ ]:
from src.training.sft import SFTTrainer, SFTConfig
from src.training.tracking import CheckpointManager

# Load best MPM checkpoint
mpm_ckpt = CheckpointManager(checkpoint_dir=f"{CKPT_DIR}/mpm", max_checkpoints=5)
mpm_checkpoint = mpm_ckpt.load_best(metric="loss", lower_is_better=True)
if mpm_checkpoint:
    print(f"Loaded MPM checkpoint (epoch {mpm_checkpoint.get('epoch', '?')})")
else:
    print("No MPM checkpoint found — SFT will train from scratch.")

# Configure SFT (large batch sizes for RTX 6000 Pro 48GB)
sft_config = SFTConfig(
    num_epochs=50,
    batch_size=32,  # 48GB VRAM allows large batches
    learning_rate=5e-5,
    checkpoint_dir=f"{CKPT_DIR}/sft",
    device="cuda" if torch.cuda.is_available() else "auto",
)

sft_ckpt = CheckpointManager(
    checkpoint_dir=f"{CKPT_DIR}/sft",
    max_checkpoints=5,
)

print(f"SFT config: epochs={sft_config.num_epochs}, batch={sft_config.batch_size}, lr={sft_config.learning_rate}")
print(f"Checkpoints: {CKPT_DIR}/sft")

In [ ]:
# Run SFT fine-tuning
# NOTE: SFT requires the tokenizer, diffusion, and constraint models to be
# wired together. This will work once trained MPM weights are available.
# For now, this cell serves as the template — uncomment when ready.

print("SFT training requires wiring tokenizer + diffusion + constraints.")
print("Run after MPM pre-training produces a checkpoint.")
print(f"SFT will load from: {CKPT_DIR}/mpm/")
print(f"SFT will save to:   {CKPT_DIR}/sft/")

# Uncomment when ready:
# sft_trainer = SFTTrainer(
#     tokenizer_model=...,
#     diffusion_model=...,
#     constraint_module=...,
#     dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     config=sft_config,
# )
# sft_trainer.train()

## 5. GRPO Quality Annealing

**Group Relative Policy Optimization** — uses the SFT model as a reference and
anneals output quality through a reward-based objective.

This is Phase C of the training pipeline. Requires an SFT checkpoint from Step 4.

In [ ]:
from src.training.grpo import GRPOTrainer, GRPOConfig

# Configure GRPO
grpo_config = GRPOConfig(
    num_iterations=1000,
    batch_size=8,  # 48GB VRAM — can go higher if needed
    group_size=8,
    checkpoint_dir=f"{CKPT_DIR}/grpo",
    device="cuda" if torch.cuda.is_available() else "auto",
    wandb_project="axon-grpo",
    wandb_enabled=True,
)

# Load SFT checkpoint as reference
sft_ckpt = CheckpointManager(checkpoint_dir=f"{CKPT_DIR}/sft", max_checkpoints=5)
sft_checkpoint = sft_ckpt.load_best(metric="loss", lower_is_better=True)
if sft_checkpoint:
    print(f"Loaded SFT checkpoint (epoch {sft_checkpoint.get('epoch', '?')})")
else:
    print("No SFT checkpoint found. GRPO needs an SFT checkpoint as reference.")

print(f"GRPO config: iterations={grpo_config.num_iterations}, batch={grpo_config.batch_size}, group={grpo_config.group_size}")

In [ ]:
# Run GRPO annealing
# NOTE: GRPO requires an SFT checkpoint as the reference model.
# This cell serves as the template — uncomment after SFT completes.

print("GRPO requires a completed SFT checkpoint as reference.")
print(f"GRPO will load from: {CKPT_DIR}/sft/")
print(f"GRPO will save to:   {CKPT_DIR}/grpo/")

# Uncomment when ready:
# grpo_trainer = GRPOTrainer.from_sft_checkpoint(
#     sft_checkpoint_path=f"{CKPT_DIR}/sft/best.pt",
#     config=grpo_config,
# )
# grpo_trainer.train()

## 6. DRL Training

**Deep Reinforcement Learning** for wall panelization and pod placement.
Uses MaskablePPO to learn optimal panel selection and product placement
policies on synthetic and ResPlan floor plans.

This is Phase D of the training pipeline. Can run independently of Phases A-C.

In [ ]:
from src.training.drl_training import DRLTrainingPipeline, DRLTrainingConfig

# Configure DRL training (large batch for RTX 6000 Pro 48GB)
drl_config = DRLTrainingConfig(
    total_timesteps=500_000,
    learning_rate=3e-4,
    batch_size=128,  # 48GB VRAM allows large batches
    n_envs=8,        # more parallel envs for faster collection
    checkpoint_dir=f"{CKPT_DIR}/drl",
    device="cuda" if torch.cuda.is_available() else "auto",
    wandb_project="axon-drl",
    wandb_enabled=True,  # Set False to disable W&B
    eval_freq=10_000,
    num_eval_episodes=20,
    resplan_path=f"{DATA_DIR}/ResPlan/ResPlan.pkl",
    use_resplan=True,  # Set False if ResPlan is not available
    seed=42,
)

print("DRL Training Configuration:")
print(f"  Timesteps:    {drl_config.total_timesteps:,}")
print(f"  Envs:         {drl_config.n_envs}")
print(f"  Batch size:   {drl_config.batch_size}")
print(f"  LR:           {drl_config.learning_rate}")
print(f"  Checkpoints:  {drl_config.checkpoint_dir}")
print(f"  ResPlan:      {drl_config.use_resplan}")
print(f"  GPU:          RTX 6000 Pro Blackwell 48GB")

In [ ]:
# Run DRL training
pipeline = DRLTrainingPipeline(config=drl_config)

try:
    pipeline.train()
    print("\nDRL training complete!")
except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Install with: !pip install sb3-contrib stable-baselines3")
finally:
    pipeline.close()

## 7. Evaluation

Run benchmarks on the trained models to assess extraction quality and DRL performance.

In [ ]:
# Evaluate DRL training results (loads from disk — survives runtime restarts)
import json
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/axon"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
LOG_DIR = Path(CKPT_DIR) / "drl" / "logs"

# Find the metrics JSONL log (contains both trained and greedy eval metrics)
metrics_files = sorted(LOG_DIR.glob("*/metrics.jsonl")) if LOG_DIR.exists() else []
if not metrics_files:
    # Fallback: check directly in the logs dir
    direct = LOG_DIR / "metrics.jsonl"
    if direct.exists():
        metrics_files = [direct]

if not metrics_files:
    print(f"No metrics log found in {LOG_DIR}")
    print("Make sure DRL training (cell 17) completed successfully.")
else:
    # Load the most recent metrics file
    log_path = metrics_files[-1]
    print(f"Loading metrics from: {log_path}\n")

    # Parse JSONL — find the final eval entry (has eval/ keys)
    eval_entry = None
    config_entry = None
    with open(log_path) as f:
        for line in f:
            entry = json.loads(line.strip())
            if entry.get("_type") == "config":
                config_entry = entry
            elif "eval/trained_reward" in entry:
                eval_entry = entry

    if eval_entry is None:
        print("Training log exists but no evaluation metrics found.")
        print("Training may not have completed. Check W&B or re-run cell 17.")
    else:
        tr = eval_entry.get("eval/trained_reward", 0)
        ts = eval_entry.get("eval/trained_spur", 0)
        twc = eval_entry.get("eval/trained_wall_coverage", 0)
        trc = eval_entry.get("eval/trained_room_coverage", 0)
        gr = eval_entry.get("eval/greedy_reward", 0)
        gs = eval_entry.get("eval/greedy_spur", 0)
        gwc = eval_entry.get("eval/greedy_wall_coverage", 0)
        grc = eval_entry.get("eval/greedy_room_coverage", 0)
        t_sec = eval_entry.get("training_time_seconds", 0)

        print("=" * 50)
        print(" DRL Evaluation Results")
        print("=" * 50)
        print(f"  Trained reward:       {tr:.2f}")
        print(f"  Trained SPUR:         {ts:.3f}")
        print(f"  Trained wall cov:     {twc:.1f}%")
        print(f"  Trained room cov:     {trc:.1f}%")
        print()
        print(f"  Greedy reward:        {gr:.2f}")
        print(f"  Greedy SPUR:          {gs:.3f}")
        print(f"  Greedy wall cov:      {gwc:.1f}%")
        print(f"  Greedy room cov:      {grc:.1f}%")
        print()
        print(f"  Reward improvement:   {tr - gr:+.2f}")
        print(f"  SPUR improvement:     {ts - gs:+.3f}")
        if t_sec:
            print(f"  Training time:        {t_sec/60:.1f} min")

    # Also show checkpoint manifest if available
    manifest_path = Path(CKPT_DIR) / "drl" / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        print(f"\n  Checkpoints saved:    {len(manifest)}")
        for ck in manifest:
            m = ck.get("metrics", {})
            metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in m.items())
            print(f"    step {ck['epoch']:>7d}: {metrics_str}")

In [ ]:
# List all saved checkpoints across training phases
from src.training.tracking import CheckpointManager

print("Saved Checkpoints:")
print("-" * 60)

for phase in ["mpm", "sft", "grpo", "drl"]:
    ckpt_dir = f"{CKPT_DIR}/{phase}"
    mgr = CheckpointManager(ckpt_dir)
    checkpoints = mgr.list_checkpoints()
    if checkpoints:
        print(f"\n  {phase.upper()} ({len(checkpoints)} checkpoints):")
        for ck in checkpoints:
            metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in ck.get("metrics", {}).items())
            print(f"    epoch {ck['epoch']:>4d}: {metrics_str}")
    else:
        print(f"\n  {phase.upper()}: no checkpoints")

---

**Next steps:**
- Download checkpoints from Google Drive for local inference
- Run the full Axon pipeline: `python -m src.pipeline report input.pdf --output-dir results/`
- Process all Capsule PDFs: `python -m src.pipeline batch CapsuleFloorPlans/Floorplans/ --output-dir results/`
- Visualize results in the Layer 1 demo notebook: `notebooks/axon_layer1_demo.ipynb`

**To pull latest code changes and re-run:**
Just restart the runtime and run from the top — cell 1 will `git pull` automatically.